# Energy Consumption Data - Exploratory Data Analysis

This notebook performs initial exploration of building energy consumption data to understand patterns, relationships, and data quality issues.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')

# Add parent directory to path
import sys
sys.path.append('..')

## 1. Load and Inspect Data

In [ ]:
# Load data using our custom data loader
from src.data.data_loader import EnergyDataLoader

loader = EnergyDataLoader(data_dir='../data')

# Load synthetic data for demonstration
energy_data = loader.load_energy_consumption_data()
weather_data = loader.load_weather_data("2023-01-01", "2023-12-31", "New York")
building_metadata = loader.load_building_metadata(energy_data['building_id'].unique())

# Integrate all datasets
df = loader.integrate_datasets(energy_data, weather_data, building_metadata)

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Number of buildings: {df['building_id'].nunique()}")

In [ ]:
# Display basic information
df.info()

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Statistical summary
df.describe()

## 2. Data Quality Analysis

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing_values,
    'Missing_Percentage': missing_percent
})

missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)

In [ ]:
# Check for duplicates
duplicates = df.duplicated(subset=['building_id', 'timestamp']).sum()
print(f"Number of duplicate records: {duplicates}")

## 3. Energy Consumption Patterns

In [ ]:
# Overall energy consumption distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(df['energy_consumption_kwh'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Energy Consumption (kWh)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Energy Consumption')

# Box plot by building type
df.boxplot(column='energy_consumption_kwh', by='building_type', ax=axes[1])
axes[1].set_xlabel('Building Type')
axes[1].set_ylabel('Energy Consumption (kWh)')
axes[1].set_title('Energy Consumption by Building Type')

plt.tight_layout()
plt.show()

In [ ]:
# Hourly consumption patterns
hourly_avg = df.groupby(df['timestamp'].dt.hour)['energy_consumption_kwh'].mean()

fig = px.line(x=hourly_avg.index, y=hourly_avg.values,
              labels={'x': 'Hour of Day', 'y': 'Average Energy Consumption (kWh)'},
              title='Average Hourly Energy Consumption Pattern')
fig.update_layout(height=400)
fig.show()

In [ ]:
# Weekly patterns
df['day_name'] = df['timestamp'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

weekly_avg = df.groupby('day_name')['energy_consumption_kwh'].mean().reindex(day_order)

fig = px.bar(x=weekly_avg.index, y=weekly_avg.values,
             labels={'x': 'Day of Week', 'y': 'Average Energy Consumption (kWh)'},
             title='Average Daily Energy Consumption by Day of Week')
fig.update_layout(height=400)
fig.show()

In [ ]:
# Monthly patterns
monthly_avg = df.groupby(df['timestamp'].dt.month)['energy_consumption_kwh'].mean()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig = px.line(x=month_names, y=monthly_avg.values,
              labels={'x': 'Month', 'y': 'Average Energy Consumption (kWh)'},
              title='Average Monthly Energy Consumption Pattern',
              markers=True)
fig.update_layout(height=400)
fig.show()

## 4. Weather Impact Analysis

In [ ]:
# Temperature vs Energy Consumption
fig = px.scatter(df.sample(10000), x='temperature_c', y='energy_consumption_kwh',
                 color='building_type', alpha=0.5,
                 labels={'temperature_c': 'Temperature (°C)',
                        'energy_consumption_kwh': 'Energy Consumption (kWh)'},
                 title='Energy Consumption vs Temperature by Building Type')
fig.update_layout(height=500)
fig.show()

In [ ]:
# Create temperature bins for better analysis
df['temp_bin'] = pd.cut(df['temperature_c'], bins=10)
temp_energy = df.groupby('temp_bin')['energy_consumption_kwh'].agg(['mean', 'std'])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[str(i) for i in temp_energy.index],
    y=temp_energy['mean'],
    error_y=dict(type='data', array=temp_energy['std']),
    mode='markers+lines',
    name='Mean ± Std'
))

fig.update_layout(
    title='Energy Consumption by Temperature Range',
    xaxis_title='Temperature Range (°C)',
    yaxis_title='Energy Consumption (kWh)',
    height=400
)
fig.show()

In [ ]:
# Correlation matrix
weather_energy_cols = ['energy_consumption_kwh', 'temperature_c', 'humidity_percent', 
                       'wind_speed_kmh', 'solar_radiation_wm2']
correlation_matrix = df[weather_energy_cols].corr()

fig = px.imshow(correlation_matrix,
                labels=dict(color="Correlation"),
                x=weather_energy_cols,
                y=weather_energy_cols,
                color_continuous_scale='RdBu_r',
                title='Correlation Matrix: Weather vs Energy Consumption')

fig.update_layout(height=500)
fig.show()

## 5. Building Characteristics Analysis

In [ ]:
# Energy consumption by building characteristics
building_summary = df.groupby('building_id').agg({
    'energy_consumption_kwh': 'mean',
    'floor_area_sqm': 'first',
    'building_type': 'first',
    'year_built': 'first',
    'energy_star_rating': 'first'
}).reset_index()

# Energy intensity (consumption per square meter)
building_summary['energy_intensity'] = (building_summary['energy_consumption_kwh'] / 
                                       building_summary['floor_area_sqm'])

# Plot energy intensity by building type
fig = px.box(building_summary, x='building_type', y='energy_intensity',
             title='Energy Intensity by Building Type',
             labels={'energy_intensity': 'Energy Intensity (kWh/m²)'})
fig.update_layout(height=400)
fig.show()

In [ ]:
# Energy Star Rating vs Consumption
fig = px.scatter(building_summary, x='energy_star_rating', y='energy_intensity',
                 color='building_type', size='floor_area_sqm',
                 title='Energy Star Rating vs Energy Intensity',
                 labels={'energy_star_rating': 'Energy Star Rating',
                        'energy_intensity': 'Energy Intensity (kWh/m²)'})
fig.update_layout(height=500)
fig.show()

## 6. Time Series Analysis

In [ ]:
# Select one building for detailed time series analysis
sample_building = df['building_id'].unique()[0]
building_ts = df[df['building_id'] == sample_building].set_index('timestamp').sort_index()

# Plot daily consumption for the building
daily_consumption = building_ts['energy_consumption_kwh'].resample('D').sum()

fig = make_subplots(rows=3, cols=1, 
                    subplot_titles=['Daily Energy Consumption', 
                                   'Weekly Rolling Average',
                                   'Monthly Aggregation'],
                    vertical_spacing=0.1)

# Daily consumption
fig.add_trace(go.Scatter(x=daily_consumption.index, y=daily_consumption.values,
                        mode='lines', name='Daily'),
              row=1, col=1)

# Weekly rolling average
weekly_avg = daily_consumption.rolling(window=7).mean()
fig.add_trace(go.Scatter(x=weekly_avg.index, y=weekly_avg.values,
                        mode='lines', name='7-day MA'),
              row=2, col=1)

# Monthly aggregation
monthly = building_ts['energy_consumption_kwh'].resample('M').sum()
fig.add_trace(go.Bar(x=monthly.index, y=monthly.values, name='Monthly'),
              row=3, col=1)

fig.update_layout(height=900, title_text=f"Time Series Analysis for Building {sample_building}")
fig.update_xaxes(title_text="Date")
fig.update_yaxes(title_text="Energy (kWh)", row=1, col=1)
fig.update_yaxes(title_text="Energy (kWh)", row=2, col=1)
fig.update_yaxes(title_text="Energy (kWh)", row=3, col=1)
fig.show()

## 7. Peak Demand Analysis

In [ ]:
# Identify peak demand periods
df['hour'] = df['timestamp'].dt.hour
df['is_peak'] = df['hour'].between(9, 17)  # Business hours

peak_comparison = df.groupby(['building_type', 'is_peak'])['energy_consumption_kwh'].mean().unstack()
peak_comparison.columns = ['Off-Peak', 'Peak']
peak_comparison['Peak_Ratio'] = peak_comparison['Peak'] / peak_comparison['Off-Peak']

fig = go.Figure()
fig.add_trace(go.Bar(name='Off-Peak', x=peak_comparison.index, y=peak_comparison['Off-Peak']))
fig.add_trace(go.Bar(name='Peak', x=peak_comparison.index, y=peak_comparison['Peak']))

fig.update_layout(title='Peak vs Off-Peak Energy Consumption by Building Type',
                  xaxis_title='Building Type',
                  yaxis_title='Average Energy Consumption (kWh)',
                  barmode='group',
                  height=400)
fig.show()

print("\nPeak to Off-Peak Ratio:")
print(peak_comparison['Peak_Ratio'].round(2))

## 8. Key Findings Summary

In [ ]:
# Calculate key statistics
total_consumption = df['energy_consumption_kwh'].sum()
avg_daily_consumption = df.groupby(df['timestamp'].dt.date)['energy_consumption_kwh'].sum().mean()
peak_hour = df.groupby(df['timestamp'].dt.hour)['energy_consumption_kwh'].mean().idxmax()
most_efficient_building_type = building_summary.groupby('building_type')['energy_intensity'].mean().idxmin()

print("=== KEY FINDINGS ===")
print(f"\n1. Total Energy Consumption: {total_consumption:,.0f} kWh")
print(f"2. Average Daily Consumption: {avg_daily_consumption:,.0f} kWh")
print(f"3. Peak Consumption Hour: {peak_hour}:00")
print(f"4. Most Energy-Efficient Building Type: {most_efficient_building_type}")
print(f"\n5. Weather Impact:")
print(f"   - Temperature correlation with consumption: {df['temperature_c'].corr(df['energy_consumption_kwh']):.3f}")
print(f"   - Optimal temperature range: {temp_energy['mean'].idxmin()}°C")
print(f"\n6. Potential Savings:")
print(f"   - Weekend vs Weekday difference: {(weekly_avg['Monday'] - weekly_avg['Sunday']) / weekly_avg['Monday'] * 100:.1f}%")
print(f"   - Peak vs Off-peak difference: {(peak_comparison['Peak'].mean() - peak_comparison['Off-Peak'].mean()) / peak_comparison['Peak'].mean() * 100:.1f}%")

## 9. Next Steps

Based on this EDA, the following steps are recommended:

1. **Feature Engineering**:
   - Create lag features for time series prediction
   - Engineer weather interaction features
   - Add cyclical encoding for temporal features

2. **Data Preprocessing**:
   - Handle missing values in weather data
   - Remove outliers in energy consumption
   - Normalize features for model training

3. **Model Development**:
   - Start with baseline time series models (ARIMA, Prophet)
   - Implement ML models (XGBoost, LightGBM)
   - Develop deep learning models (LSTM) for sequence prediction

4. **Optimization Opportunities**:
   - Focus on peak hour load management
   - Implement temperature-based HVAC optimization
   - Target high energy intensity buildings